> **中文版阅读说明：** 本文件按原单元格顺序翻译。执行代码、英文实验数据、标签、模型提示词和历史输出均予以保留；英文数据与提示词另附中文对照，避免翻译改变实验条件。本次未重新运行模型。标有“译注”的文字为补充说明，其余是原文翻译。原模板仍有“植物示例”“REPLACE ME”等说明，但当前 DATA 已替换为养猫咨询；这些模板描述不代表当前数据内容。


In [ ]:
# A1 · Part 1——选择合适的工具。入门 Notebook。
# PE6201 · 新兴人工智能技术。由上到下运行，再替换占位领域。

# A1 · Part 1 — 选择合适的工具（入门 Notebook）

**Part 1 要求你做什么。** 在**你自己的领域**选择一个分类任务（Classification），将输入分到两个或更多类别。用**两种方法**实现：一种是在带标签样本上训练 **scikit-learn 分类器**；另一种是仅通过提示词调用**基础模型（Foundation Model）API**，完全不进行训练。比较两者的**准确率（Accuracy）、成本（Cost）、延迟（Latency）和投入（Effort）**，然后回答：*你实际会部署哪一种？业务规模达到多大时，你会改变选择？*

**这个 Notebook 是什么。** 它已在一个**占位示例领域**（室内植物照护咨询）上实现完整比较。必须修改的位置用 `### REPLACE ME` 标记，其余位置无需编辑。

**占位领域不是推荐选题。** 不要提交室内植物示例，请换成你熟悉的领域，并参考作业说明中的选题指导。

**成本。** 原文估计：使用默认模型进行约 60 次简短 API 调用，费用远低于 1 美分。末尾仍会统计 Token，因为成本是四项评测内容之一。这是原文件中的估计，不是本次核实的当前价格。

**此 Notebook 的交付物：** 已运行且显示输出的 Notebook、一张比较表，以及约 400 词的分析。

> **如果某个单元格无法运行：** 编写时所用的库版本、模型名称和服务商行为可能发生变化，因此应将本文件视为可修订的工作稿。**先阅读报错，再尝试自行修复：** 调试也是作业的一部分。随后在课堂上或通过邮件报告问题，附上对应单元格、完整错误和尝试过的方法，以便修正共享版本。如果修复改变了 Notebook 测量的内容，请在报告中说明。


### 环境设置（Setup）

In [ ]:
%pip -q install openai scikit-learn

### 连接至 OpenRouter
与第 1、2、3 次课的 Notebook 使用相同方式。密钥是单独发给你的。
**不要把密钥直接写进代码单元格。** 代码会从 Colab Secret、环境变量或隐藏输入框读取它。

In [ ]:
import os, getpass, time, json
from openai import OpenAI

def _get_openrouter_key():
    k = os.environ.get('MY_PRIVATE_OPENROUTER_KEY') or os.environ.get('OPENROUTER_API_KEY')
    if k: return k
    try:
        from google.colab import userdata
        k = userdata.get('OPENROUTER_API_KEY')
        if k: return k
    except Exception:
        pass
    return getpass.getpass('Paste your OpenRouter API key (hidden): ')

API_KEY = _get_openrouter_key()
client  = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=API_KEY)

# 可调选项：尝试密钥能够访问的任意模型。廉价模型也值得公平比较——
# 你的论点可以是小模型已经足够好。
#   'openai/gpt-4o-mini' (default)  ·  'anthropic/claude-3.5-haiku'
#   'qwen/qwen-2.5-7b-instruct'     ·  'deepseek/deepseek-chat'
GEN_MODEL = 'openai/gpt-4o-mini'

TOKENS = {'in': 0, 'out': 0}

def generate(prompt: str, system: str = None, max_new_tokens: int = 40,
             temperature: float = 0.0) -> str:
    """原文说明：temperature=0 用于减少随机性，便于比较两种方法。
       译注：设置为 0 不代表服务端输出必然完全确定。"""
    msgs = ([{'role': 'system', 'content': system}] if system else []) + \
           [{'role': 'user', 'content': prompt}]
    r = client.chat.completions.create(model=GEN_MODEL, messages=msgs,
                                       temperature=temperature, max_tokens=max_new_tokens)
    if getattr(r, 'usage', None):
        TOKENS['in']  += r.usage.prompt_tokens
        TOKENS['out'] += r.usage.completion_tokens
    return r.choices[0].message.content.strip()

print('connected · model:', GEN_MODEL, '·', generate('Reply with exactly the word: ready'))

Paste your OpenRouter API key (hidden): ··········
connected · model: openai/gpt-4o-mini · ready


### 代码中的英文文字对照
- `Paste your OpenRouter API key (hidden)`：粘贴 OpenRouter API 密钥（隐藏输入）。
- `Reply with exactly the word: ready`：只回复 ready 这个词，用于检查连接。
- `connected · model`：已连接·模型。
- `TOKENS` 的 `in` / `out`：输入/输出 Token。

### 检查剩余额度——免费，此调用不扣费
你的分配额度**固定，不会追加**，请及时查看，不要到最后才发现额度用尽。`usage` 是已使用金额，`limit` 是额度上限，两者之差就是剩余额度。每次开始和结束工作时都运行一次。

In [ ]:
import urllib.request, json as _json

def credit_check():
    """向 OpenRouter 查询此密钥的剩余额度，此查询不收费。"""
    try:
        req = urllib.request.Request('https://openrouter.ai/api/v1/key',
                                     headers={'Authorization': f'Bearer {API_KEY}'})
        d = _json.load(urllib.request.urlopen(req, timeout=20))['data']
        used, cap = d.get('usage'), d.get('limit')
        if cap is None:
            print(f'spent so far: ${used:.4f}   (no cap set on this key)')
        else:
            print(f'spent ${used:.4f} of ${cap:.2f}  ->  ${cap - used:.4f} remaining')
        rl = d.get('rate_limit') or {}
        if rl:
            print(f"rate limit: {rl.get('requests')} requests per {rl.get('interval')}")
    except Exception as e:
        print('could not check credit (not a problem, carry on):', str(e)[:120])

credit_check()

spent $0.0005 of $10.00  ->  $9.9995 remaining
rate limit: -1 requests per 10s


### 额度输出文字对照
- `spent so far`：截至目前已花费。
- `no cap set on this key`：此密钥没有设置额度上限。
- `spent … of … → … remaining`：总额度中已使用多少、还剩多少。
- `rate limit`：请求速率限制。
- `could not check credit (not a problem, carry on)`：无法检查额度（可以继续）。

---
## 0 · 需要掌握的六个术语——不会编程可以从这里开始

完成本任务不需要写很多 Python。阅读说明、替换数据、运行单元格，再观察四项指标。

| 术语 | 通俗解释 |
|---|---|
| **分类（Classification）** | 把每条输入分到预先确定的某个类别：它属于这三个“桶”中的哪一个？ |
| **标签（Label）** | 一条样本的正确类别，由人（也就是你）提供。 |
| **训练（Training）** | 给程序展示许多带标签的样本，让它学会预测新样本的标签。 |
| **准确率（Accuracy）** | 测试样本中预测正确的比例，取值为 0 到 1。 |
| **延迟（Latency）** | 完成一次预测所需的时间；这里用毫秒/条表示。 |
| **基线（Baseline）** | 简单、廉价的方法能得到的成绩，例如一直猜最常见的类别。若结果不能超过基线，就无法体现方法的价值。 |

**Part 1 的重点并不是让某种方法获胜，**而是说明选择哪种方法、证据是什么，以及达到什么规模时选择会改变。

---
## 1 · 你的数据（`### REPLACE ME`：请替换）

需要带标签的样本：一段短文本，以及它的正确类别。

**数量：** 60–120 条足够。少于约 40 条时，准确率波动较大，不便比较。

**类别数量：** 两类或三类。超过四类可能会花费大量标注时间。

**来源：** 根据自己熟悉的领域编写、从已有材料导出，或手动收集。**不要让 AI 助手凭空编造数据集。** 重点是你能辨别正确答案和错误答案，因此需要真正熟悉数据。如果必须生成样本，应在报告中披露，并解释由此造成的代价或局限。

**保持均衡：** 如果 90% 的样本属于同一类，一直猜该类也能获得 90% 准确率。各类数量应大致均衡，并与下方输出的**多数类基线（Majority-class Baseline）**比较。

In [ ]:
### REPLACE ME——替换为自己的类别，建议两类或三类，使用领域内的名称。
LABELS = ['routine_care', 'behaviour', 'health_concern']

### REPLACE ME——替换为自己的带标签样本：(文本, 标签)。
# 原模板的占位数据是室内植物照护消息。自己的数据可以是分诊记录、支持工单、
# 比赛报告、食谱步骤、歌词、招聘广告等领域内真实产生的文本。
DATA = [
    ("My cat frequently bites my arms, jumps on me and kicks me even when I am not interacting with her. However, she remains affectionate and sleeps beside me at night. Why does she behave aggressively toward me?", "behaviour"),
    ("My cat meows loudly for hours whenever I lie down to sleep. He behaves normally during the day, and additional playtime, food and toys have not stopped the nighttime vocalisation. How can I reduce this behaviour?", "behaviour"),
    ("My 10-week-old kitten refuses to use any of the three litter boxes and defecates elsewhere, even though I use the same litter as his previous owner. How can I encourage him to use the litter box?", "behaviour"),
    ("My cat continues to scratch the couch and bed despite having several scratching posts. How can I redirect this behaviour?", "behaviour"),
    ("My newly adopted cat hides throughout the day and only comes out at night to eat and use the litter box. How can I help her feel safe?", "behaviour"),
    ("My cat frequently bites during interaction and sometimes breaks the skin. How can I discourage this behaviour?", "behaviour"),
    ("My cat bites my feet or legs when he wants attention or a response. How can I change this behaviour?", "behaviour"),
    ("My cat asks to be petted but then bites and immediately asks for more attention. Why does this cycle occur?", "behaviour"),
    ("My normally gentle cat suddenly bit my heel and growled, then hid from me. What could have triggered this aggression?", "behaviour"),
    ("My cat scratches household furniture even though scratching items are available. How can I redirect this behaviour?", "behaviour"),
    ("My cat repeatedly scratches one area of carpet and ignores the available scratchers. How can I redirect her?", "behaviour"),
    ("My cat scratches door frames instead of her scratching posts. How can I change this preference?", "behaviour"),
    ("My cat has begun urinating and defecating outside the litter box despite no environmental or medical change. How can I address this?", "behaviour"),
    ("My cat refuses several types of litter and rarely uses the box despite medical causes being ruled out. What should I try next?", "behaviour"),
    ("My cat repeatedly chews electrical cords, including plugged-in ones. How can I stop this dangerous behaviour?", "behaviour"),
    ("My cats are moving into a home with a kitchen for the first time. How can I discourage counter jumping?", "behaviour"),
    ("My cat immediately hides whenever visitors arrive, although she is relaxed with household members. How can I help her?", "behaviour"),
    ("My affectionate cat becomes silent and hides whenever someone visits. How can I socialise her without forcing interaction?", "behaviour"),
    ("My cat is affectionate with me but hides for as long as other people remain in the house. Can this fear be reduced?", "behaviour"),
    ("My cat meows for hours every morning despite being ignored and not being fed during the meowing. How can I change this habit?", "behaviour"),
    ("My cat wakes at the same time every night and meows loudly for attention. Why does this happen, and how can I reduce it?", "behaviour"),
    ("My cat leaves the bedroom and cries downstairs each night after we go to bed. What could be maintaining this behaviour?", "behaviour"),
    ("When I close the bedroom door, my cat meows constantly and scratches the door and carpet. How can I change this response?", "behaviour"),
    ("My cats repeatedly fight despite attempts to manage their interactions. How should I safely reduce the conflict?", "behaviour"),
    ("My cat panics when picked up or placed near her carrier. How can I gradually reduce her carrier fear?", "behaviour"),
    ("My cat leaves strong-smelling urine on areas outside the litter box. Could this be territorial marking, and how can I address it?", "behaviour"),
    ("My kitten avoids the automatic litter box and urinates elsewhere. How can I determine whether the box design or placement is causing this?", "behaviour"),
    ("My cat frequently pounces on and attacks me during ordinary interactions. How can I identify and reduce the trigger?", "behaviour"),
    ("My newly adopted shelter cat has hidden for three weeks and hisses when approached. How can I help her adjust without pressure?", "behaviour"),
    ("My cat repeatedly wakes me during the early morning. How can I adjust the routine that may be reinforcing this behaviour?", "behaviour"),
    ("My cat has a sensitive stomach and continues to vomit despite changes to her food, feeding bowls and digestive supplements. She sometimes vomits several times a day. Could this cause long-term health problems, and should I contact a veterinarian?", "health_concern"),
    ("I accidentally stepped on my four-month-old kitten's paw, and he is now limping. Does he require veterinary attention?", "health_concern"),
    ("My cat may have swallowed a small piece of plastic packaging. He is not showing symptoms, but should I contact a veterinarian?", "health_concern"),
    ("My cat began panting after about twenty minutes of active play. This has never happened before. Should I be concerned?", "health_concern"),
    ("My cat has barely eaten or drunk anything for four days since moving to a new home. Could severe stress be affecting her health, and should I contact a veterinarian?", "health_concern"),
    ("My older cat's diarrhoea repeatedly returns after treatment changes. Does this require further veterinary assessment?", "health_concern"),
    ("My kitten has had soft or watery stool with traces of blood for several days. Should I seek veterinary care?", "health_concern"),
    ("My cat has not defecated for five days following dental surgery, although he otherwise seems normal. Should I contact the veterinarian urgently?", "health_concern"),
    ("My cat has limped on one front leg for several weeks despite normal appetite and activity. Does he need a veterinary examination?", "health_concern"),
    ("I accidentally cut my cat's nail too short and caused bleeding. What signs would indicate that veterinary care is needed?", "health_concern"),
    ("My cat suddenly holds one paw off the ground after a nail trim. Could the paw be injured?", "health_concern"),
    ("My cat swallowed a piece of string but currently appears normal. Should I contact a veterinarian?", "health_concern"),
    ("My cat ate an entire mouse. What health risks should I monitor, and should I contact a veterinarian?", "health_concern"),
    ("My young cat sometimes breathes with his mouth open after running around. Is this a health concern?", "health_concern"),
    ("My cat develops noisy, heavy breathing while relaxed and falling asleep. Should I show this to a veterinarian?", "health_concern"),
    ("My cat has repeatedly sneezed, coughed and gagged since yesterday. Does she need a veterinary examination?", "health_concern"),
    ("My cat has continued sneezing intermittently for several weeks. When does this require veterinary assessment?", "health_concern"),
    ("My cat's urine contains blood again after two symptom-free weeks. Does she need urgent veterinary care?", "health_concern"),
    ("My cat repeatedly strains to urinate but produces only small amounts. Does this require emergency veterinary attention?", "health_concern"),
    ("My cat repeatedly develops eye discharge after sleeping. When should recurring discharge be checked by a veterinarian?", "health_concern"),
    ("Both of my cat's eyes now water daily, although the fluid is clear. Should I arrange a veterinary examination?", "health_concern"),
    ("My cat suddenly began drooling heavily during sleep. Should this new symptom be examined by a veterinarian?", "health_concern"),
    ("Two cats in the same home have developed blood in their stool. Should they receive veterinary assessment?", "health_concern"),
    ("My cat has gradually lost weight despite no obvious finding on a basic examination. Should further tests be considered?", "health_concern"),
    ("My cat has refused food for five days after dental surgery and is becoming weak. Does he need urgent reassessment?", "health_concern"),
    ("My cat is unable to eat or drink normally and has become lethargic. What urgent professional care is needed?", "health_concern"),
    ("My cat has been unusually lethargic for several days despite continuing to eat and drink. Should he be examined?", "health_concern"),
    ("My cat is sleeping constantly and eating and drinking very little after neutering. Are these postoperative signs concerning?", "health_concern"),
    ("Dark crusty discharge repeatedly returns around my cat's eyes. Should this be checked by a veterinarian?", "health_concern"),
    ("My cat developed persistent diarrhoea after switching to a prescribed diet. Should I contact the veterinarian about changing it?", "health_concern"),
    ("My cat's litter box is kept in the bedroom and is sometimes left unscooped for four days, causing a strong smell. How often should a litter box normally be scooped?", "routine_care"),
    ("I am preparing my apartment for my first cat. How should I arrange the litter box, feeding area, resting space and scratching facilities to create a safe living environment?", "routine_care"),
    ("My long-haired cat cannot properly clean the fur on his back, which has become clumpy and matted. He becomes frightened and aggressive when I try to wash him. How can I clean and groom him safely?", "routine_care"),
    ("I am preparing to adopt a cat. Should an adult cat receive two larger meals or three smaller meals each day?", "routine_care"),
    ("My cat becomes frightened and struggles whenever I try to trim his nails. How can I trim them safely with less stress?", "routine_care"),
    ("My cat refuses to let me hold his paws, but his nails are becoming long and sharp. How can I trim them safely?", "routine_care"),
    ("My cat dislikes being held and will not let me brush her teeth. How can I introduce tooth brushing gradually?", "routine_care"),
    ("My cat has mild dental buildup but resists thorough brushing. Are there safe routine dental-care alternatives?", "routine_care"),
    ("How should I clean my cat's ears safely, and which parts of the ear should I avoid touching?", "routine_care"),
    ("My cat struggles and scratches during baths. How can I bathe him safely while reducing stress and avoiding injury?", "routine_care"),
    ("My long-haired cat develops mats despite regular grooming. How can I remove them safely and prevent new mats?", "routine_care"),
    ("My cat prefers running tap water. What should I consider when choosing and maintaining a drinking fountain?", "routine_care"),
    ("What cleaning routine can keep a litter-box area hygienic and reduce odour?", "routine_care"),
    ("How often should I scoop waste, replace litter and wash my kitten's litter box?", "routine_care"),
    ("My home is small. Where should I place the litter box so it remains quiet, accessible and separate from food?", "routine_care"),
    ("I must keep the litter box in my bedroom. How can I control odour while keeping it comfortable for my cat?", "routine_care"),
    ("How can I cat-proof a studio apartment when most areas cannot be closed off?", "routine_care"),
    ("What should a safe room for a newly adopted cat contain, and how should it be arranged?", "routine_care"),
    ("How long should a newly arrived cat remain in one room before gradually exploring the rest of the home?", "routine_care"),
    ("What should I prepare to make my apartment safe and welcoming when a cat arrives for the first time?", "routine_care"),
    ("How often should wet food be offered when caring for cats of different ages?", "routine_care"),
    ("I recently adopted an adult cat and received conflicting feeding advice. How should I establish a consistent meal schedule?", "routine_care"),
    ("How can I interpret feeding guidelines and calculate a suitable daily portion for my cats?", "routine_care"),
    ("How should I divide my adult cat's daily food between wet and dry portions?", "routine_care"),
    ("What daily schedule should I use when feeding both wet and dry food?", "routine_care"),
    ("How can I combine wet and dry food affordably while ensuring my large cat receives enough food?", "routine_care"),
    ("Can an adult cat safely stay home alone overnight if food, water and a clean litter box are available?", "routine_care"),
    ("What care arrangements are necessary if a five-week-old kitten would otherwise be alone during a full workday?", "routine_care"),
    ("What care and check-in arrangements are needed when leaving two cats at home during a four-day trip?", "routine_care"),
    ("Which hygiene tasks should form part of a regular care routine for an adult cat?", "routine_care"),
]
# 原模板样本少以便快速运行；自己的数据建议达到 60–120 条。

texts  = [t for t, _ in DATA]
labels = [l for _, l in DATA]
from collections import Counter
counts = Counter(labels)
majority = counts.most_common(1)[0]
print(f'{len(DATA)} examples · {dict(counts)}')
print(f'MAJORITY-CLASS BASELINE: always guess "{majority[0]}" -> '
      f'{majority[1]/len(DATA):.1%} accuracy. Both approaches must beat this to mean anything.')

90 examples · {'behaviour': 30, 'health_concern': 30, 'routine_care': 30}
MAJORITY-CLASS BASELINE: always guess "behaviour" -> 33.3% accuracy. Both approaches must beat this to mean anything.


### 90 条实验数据：中文对照（阅读用）

标签：`behaviour`＝行为问题；`health_concern`＝健康担忧；`routine_care`＝日常照护。标签忠实保留原标注，本次未重新判断其正确性。

| 编号 | 原标签 | 中文咨询 |
|---|---|---|
| 1 | `behaviour` | 我的猫经常咬我的手臂、扑到我身上并踢我，即使我没有与她互动。不过她仍很亲人，晚上睡在我旁边。为什么她会对我表现出攻击性？ |
| 2 | `behaviour` | 每当我躺下睡觉，我的猫就大声叫好几个小时。白天他表现正常，增加玩耍时间、食物和玩具都没能阻止夜间叫声。我该如何减少这种行为？ |
| 3 | `behaviour` | 我的十周龄幼猫不愿使用任何一个猫砂盆（共有三个），总在别处排便，尽管猫砂与前主人用的一样。怎样鼓励他使用猫砂盆？ |
| 4 | `behaviour` | 尽管有好几个猫抓柱，我的猫仍抓沙发和床。怎样将这种行为引导到合适的地方？ |
| 5 | `behaviour` | 我新领养的猫白天一直躲着，只有夜里出来吃饭、使用猫砂盆。怎样帮助她获得安全感？ |
| 6 | `behaviour` | 我的猫互动时经常咬人，有时会咬破皮肤。怎样减少这种行为？ |
| 7 | `behaviour` | 我的猫想获得关注或回应时，会咬我的脚或腿。怎样改变这种行为？ |
| 8 | `behaviour` | 我的猫要求抚摸，却又咬我，随后马上再次索要关注。为什么会反复这样？ |
| 9 | `behaviour` | 平时温顺的猫突然咬我的脚后跟并低吼，随后躲开我。什么可能触发了这次攻击？ |
| 10 | `behaviour` | 虽然有猫抓用品，我的猫仍抓家里的家具。怎样引导这种行为？ |
| 11 | `behaviour` | 我的猫反复抓同一块地毯，却不使用现有猫抓用品。怎样引导她？ |
| 12 | `behaviour` | 我的猫不抓猫抓柱，反而抓门框。怎样改变这种偏好？ |
| 13 | `behaviour` | 在环境和健康状况没有变化的情况下，我的猫开始在猫砂盆外大小便。怎样处理？ |
| 14 | `behaviour` | 已经排除健康原因，但我的猫不接受几种猫砂，很少使用猫砂盆。下一步该尝试什么？ |
| 15 | `behaviour` | 我的猫反复啃电线，包括正在通电的电线。怎样制止这种危险行为？ |
| 16 | `behaviour` | 我的猫第一次搬进有厨房的家。怎样让它们减少跳上操作台的行为？ |
| 17 | `behaviour` | 我的猫在家人面前很放松，但客人一来就立即躲起来。怎样帮助她？ |
| 18 | `behaviour` | 我亲人的猫每当有人来访就不叫了，并躲起来。怎样在不强迫互动的情况下帮助她适应他人？ |
| 19 | `behaviour` | 我的猫对我很亲近，但只要其他人仍在家里，她就一直躲着。这种害怕能减轻吗？ |
| 20 | `behaviour` | 我的猫每天早上叫几个小时，即使我不理她，也不在叫的时候喂食。怎样改变这个习惯？ |
| 21 | `behaviour` | 我的猫每天夜里在同一时间醒来，大声叫着求关注。为什么这样，怎样减少？ |
| 22 | `behaviour` | 我们每晚上床后，猫就离开卧室，到楼下叫。什么因素可能让这种行为持续？ |
| 23 | `behaviour` | 我关上卧室门时，猫就不停叫，并抓门和地毯。怎样改变这种反应？ |
| 24 | `behaviour` | 尽管尝试管理互动，我的猫仍反复打架。怎样安全地减少冲突？ |
| 25 | `behaviour` | 我的猫被抱起或靠近航空箱时会惊慌。怎样逐步减轻她对航空箱的恐惧？ |
| 26 | `behaviour` | 我的猫在猫砂盆外留下气味很重的尿。这可能是领地标记吗？怎样处理？ |
| 27 | `behaviour` | 我的幼猫避开自动猫砂盆，在别处排尿。怎样判断是否由猫砂盆设计或摆放位置造成？ |
| 28 | `behaviour` | 我的猫在日常互动中经常扑向并攻击我。怎样找出并减少触发因素？ |
| 29 | `behaviour` | 新领养的收容所猫已经躲了三周，靠近时会嘶叫。怎样在不施压的情况下帮助她适应？ |
| 30 | `behaviour` | 我的猫反复在清晨叫醒我。怎样调整可能强化这种行为的日常安排？ |
| 31 | `health_concern` | 我的猫肠胃敏感，换了食物、食盆和消化补充剂后仍持续呕吐，有时一天吐好几次。这会造成长期健康问题吗？我应该联系兽医吗？ |
| 32 | `health_concern` | 我不小心踩到四个月大幼猫的爪子，他现在跛行。需要兽医处理吗？ |
| 33 | `health_concern` | 我的猫可能吞下了一小片塑料包装。他目前没有症状，但我应该联系兽医吗？ |
| 34 | `health_concern` | 我的猫剧烈玩耍约二十分钟后开始喘气，以前从未这样。我该担心吗？ |
| 35 | `health_concern` | 搬到新家后，我的猫四天几乎没吃没喝。严重应激会影响她的健康吗？应该联系兽医吗？ |
| 36 | `health_concern` | 我的老年猫在调整治疗后仍反复腹泻。需要进一步接受兽医评估吗？ |
| 37 | `health_concern` | 我的幼猫连续几天排软便或水样便，并带少量血迹。应该寻求兽医帮助吗？ |
| 38 | `health_concern` | 牙科手术后，我的猫五天没有排便，但其他表现正常。应该紧急联系兽医吗？ |
| 39 | `health_concern` | 我的猫一条前腿跛行几周，但食欲和活动正常。需要兽医检查吗？ |
| 40 | `health_concern` | 我不小心把猫指甲剪得太短，导致出血。哪些表现说明需要兽医帮助？ |
| 41 | `health_concern` | 剪完指甲后，我的猫突然将一只爪子抬离地面。爪子可能受伤了吗？ |
| 42 | `health_concern` | 我的猫吞了一段线，但目前看起来正常。应该联系兽医吗？ |
| 43 | `health_concern` | 我的猫吃下了一整只老鼠。应观察哪些健康风险，应该联系兽医吗？ |
| 44 | `health_concern` | 我的年轻猫跑动后有时张嘴呼吸。这是健康问题吗？ |
| 45 | `health_concern` | 我的猫放松、快睡着时呼吸沉重并有声音。应该让兽医看看吗？ |
| 46 | `health_concern` | 从昨天开始，我的猫反复打喷嚏、咳嗽和干呕。需要兽医检查吗？ |
| 47 | `health_concern` | 我的猫已断断续续打喷嚏几周。什么情况下需要兽医评估？ |
| 48 | `health_concern` | 两周没有症状后，我的猫尿中再次出现血。需要紧急接受兽医照护吗？ |
| 49 | `health_concern` | 我的猫反复用力排尿，但每次只有少量尿。需要兽医急诊吗？ |
| 50 | `health_concern` | 我的猫睡醒后反复出现眼部分泌物。什么时候应该由兽医检查？ |
| 51 | `health_concern` | 我的猫两只眼睛现在每天流泪，液体是透明的。应该安排兽医检查吗？ |
| 52 | `health_concern` | 我的猫睡觉时突然大量流口水。这个新症状应该由兽医检查吗？ |
| 53 | `health_concern` | 同一个家里的两只猫都出现便血。应该接受兽医评估吗？ |
| 54 | `health_concern` | 基础检查没有明显发现，但我的猫逐渐消瘦。应考虑进一步检查吗？ |
| 55 | `health_concern` | 我的猫牙科手术后五天拒绝进食，并变得虚弱。需要紧急复查吗？ |
| 56 | `health_concern` | 我的猫无法正常吃喝，并变得精神不振。需要哪些紧急专业照护？ |
| 57 | `health_concern` | 我的猫几天来异常精神不振，但仍吃喝。应该接受检查吗？ |
| 58 | `health_concern` | 绝育后，我的猫一直睡觉，吃喝很少。这些术后表现令人担忧吗？ |
| 59 | `health_concern` | 我的猫眼周反复出现深色结痂分泌物。应该让兽医检查吗？ |
| 60 | `health_concern` | 换成处方饮食后，我的猫持续腹泻。应联系兽医讨论调整饮食吗？ |
| 61 | `routine_care` | 猫砂盆放在卧室，有时四天没有铲便，气味很重。一般应多久铲一次？ |
| 62 | `routine_care` | 我正为第一只猫准备公寓。怎样安排猫砂盆、喂食区、休息空间和猫抓设施，营造安全环境？ |
| 63 | `routine_care` | 我的长毛猫无法清理背部，毛发已经结团、打结。我尝试给他洗澡时，他会害怕并攻击我。怎样安全清洁和梳理？ |
| 64 | `routine_care` | 我准备领养一只猫。成年猫每天应该吃两顿较大的餐，还是三顿较小的餐？ |
| 65 | `routine_care` | 每次剪指甲，我的猫都会害怕并挣扎。怎样安全剪指甲，并减少压力？ |
| 66 | `routine_care` | 我的猫不让我握他的爪子，但指甲越来越长、越来越尖。怎样安全修剪？ |
| 67 | `routine_care` | 我的猫不喜欢被抱，也不让我刷牙。怎样逐步引入刷牙？ |
| 68 | `routine_care` | 我的猫有轻微牙垢，但抗拒彻底刷牙。有安全的日常牙齿护理替代方式吗？ |
| 69 | `routine_care` | 怎样安全清洁猫耳朵？哪些部位应避免触碰？ |
| 70 | `routine_care` | 我的猫洗澡时挣扎并抓人。怎样安全给他洗澡，减少压力并避免受伤？ |
| 71 | `routine_care` | 尽管经常梳理，我的长毛猫仍会打结。怎样安全去除并预防毛结？ |
| 72 | `routine_care` | 我的猫喜欢流动的自来水。选择和维护饮水机时应考虑什么？ |
| 73 | `routine_care` | 怎样的清洁安排能保持猫砂盆区域卫生，并减少气味？ |
| 74 | `routine_care` | 幼猫猫砂盆应多久铲便、换猫砂和清洗一次？ |
| 75 | `routine_care` | 我家很小。猫砂盆应放在哪里，才能安静、容易使用，并与食物分开？ |
| 76 | `routine_care` | 我必须把猫砂盆放在卧室。怎样控制气味，同时让猫舒适使用？ |
| 77 | `routine_care` | 单间公寓大部分区域无法隔开，怎样做好防猫意外的安全布置？ |
| 78 | `routine_care` | 给新领养猫准备的安全房间应有哪些物品，怎样布置？ |
| 79 | `routine_care` | 新来的猫应在一个房间待多久，再逐步探索家里其他区域？ |
| 80 | `routine_care` | 第一只猫到家前，应准备什么，让公寓安全又舒适？ |
| 81 | `routine_care` | 照护不同年龄的猫时，应多久提供一次湿粮？ |
| 82 | `routine_care` | 我最近领养了成年猫，收到互相矛盾的喂食建议。怎样建立一致的用餐安排？ |
| 83 | `routine_care` | 怎样理解喂食指南，并计算适合猫的每日食量？ |
| 84 | `routine_care` | 成年猫每天的食物应如何分配为湿粮和干粮？ |
| 85 | `routine_care` | 同时喂湿粮和干粮时，应采用怎样的每日安排？ |
| 86 | `routine_care` | 怎样以经济的方式搭配湿粮和干粮，同时确保体型较大的猫吃得足够？ |
| 87 | `routine_care` | 如果有食物、水和干净猫砂盆，成年猫可以安全独自在家过夜吗？ |
| 88 | `routine_care` | 如果五周龄幼猫在整个工作日都将独处，需要哪些照护安排？ |
| 89 | `routine_care` | 出门旅行四天、将两只猫留在家里时，需要怎样的照护和探访安排？ |
| 90 | `routine_care` | 成年猫的日常照护应包含哪些卫生清洁任务？ |

输出中的 `examples`＝样本数；`MAJORITY-CLASS BASELINE`＝多数类基线；`always guess`＝始终预测；`accuracy`＝准确率。

### 数据集划分：训练集（Training Set）与测试集（Test Set）
分类器在**训练集**上学习，然后在从未见过的**测试集**上评分。API 方法完全不训练，但使用**完全相同的测试样本**评分，这样两种准确率才可以比较。

原文用“训练部分/测试部分”解释划分；实际下方代码的 `test_size=0.35` 表示测试集占 35%，不是各占一半。

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.35, random_state=0, stratify=labels)
print(f'{len(X_train)} training examples · {len(X_test)} test examples')

58 training examples · 32 test examples


输出对照：`training examples`＝训练样本；`test examples`＝测试样本。

---
## 2 · 方法 A——经过训练的分类器

**TF-IDF（词频—逆文档频率）**根据文本中出现的词将消息转换成数值，并赋予权重：常见词（如 `the`、`it`）权重较低，具有区分度的词（如 `webbing`：网状物，`soggy`：湿透的）权重较高。

**逻辑回归（Logistic Regression）**随后学习哪些词更倾向于对应哪个类别。原文强调，这里依靠词汇统计，而不是像人一样理解含义。这种技术已有数十年历史，能够在笔记本电脑上运行，每次预测不产生 API 调用费用。

*译注：本地预测仍有计算和维护成本；“免费”应理解为没有按次收取的模型 API 费用。*

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report

# 可调选项：ngram_range=(1,2) 同时统计连续词对（如 soft mushy），通常有帮助。
clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000))

t0 = time.perf_counter()
clf.fit(X_train, y_train)
train_secs = time.perf_counter() - t0

t0 = time.perf_counter()
pred_clf = clf.predict(X_test)
clf_latency_ms = (time.perf_counter() - t0) / len(X_test) * 1000

acc_clf = accuracy_score(y_test, pred_clf)
print(f'trained in {train_secs:.2f}s · accuracy {acc_clf:.1%} · {clf_latency_ms:.2f} ms per item\n')
print(classification_report(y_test, pred_clf, zero_division=0))

trained in 0.10s · accuracy 87.5% · 0.12 ms per item

                precision    recall  f1-score   support

     behaviour       0.83      0.91      0.87        11
health_concern       0.83      0.91      0.87        11
  routine_care       1.00      0.80      0.89        10

      accuracy                           0.88        32
     macro avg       0.89      0.87      0.88        32
  weighted avg       0.89      0.88      0.88        32



输出对照：`trained in`＝训练耗时；`accuracy`＝准确率；`ms per item`＝每条耗时（毫秒）。分类报告中：`precision`＝精确率，`recall`＝召回率，`f1-score`＝F1 分数，`support`＝该类真实样本数，`macro avg`＝各类等权平均，`weighted avg`＝按各类样本数加权平均。

---
## 3 · 方法 B——不训练的基础模型（Foundation Model）

完全不使用训练数据。用文字说明任务，再逐条将消息交给模型。

阅读提示词时注意两点：第一，**限制输出格式**——只能从列表中选择一个标签。开放式提问可能得到一整段回答，无法直接自动评分。第二，下方提示词**刻意保持简单**，改进提示词是后面可以尝试的调优选项。

In [ ]:
### REPLACE ME——用一句话描述自己的任务，保留输出约束。
SYSTEM = (
    "You classify cat owners' consultation messages by their main type of concern.\n"
    f"Reply with exactly one label from this list: {', '.join(LABELS)}.\n"
    "No explanation, no punctuation, no other words."
)

def classify_by_api(text: str) -> str:
    out = generate(f'Message: "{text}"\nCategory:', system=SYSTEM, max_new_tokens=6)
    out = out.strip().strip('.').lower()
    return out if out in LABELS else 'UNPARSEABLE'   # 无法解析的输出也计入失败，不要隐藏。

t0 = time.perf_counter()
pred_api = [classify_by_api(t) for t in X_test]
api_latency_ms = (time.perf_counter() - t0) / len(X_test) * 1000

acc_api = accuracy_score(y_test, pred_api)
bad = sum(1 for p in pred_api if p == 'UNPARSEABLE')
print(f'accuracy {acc_api:.1%} · {api_latency_ms:.0f} ms per item · {bad} unparseable replies\n')
print(classification_report(y_test, pred_api, zero_division=0))

accuracy 93.8% · 899 ms per item · 0 unparseable replies

                precision    recall  f1-score   support

     behaviour       0.85      1.00      0.92        11
health_concern       1.00      1.00      1.00        11
  routine_care       1.00      0.80      0.89        10

      accuracy                           0.94        32
     macro avg       0.95      0.93      0.94        32
  weighted avg       0.95      0.94      0.94        32



### 模型提示词中文对照（执行时仍使用原英文）
你根据主要关注的问题类型，对猫主人的咨询进行分类。
仅从以下列表中选择一个标签回复：routine_care、behaviour、health_concern。
不要解释，不要标点，不要其他文字。

用户提示：`Message`＝消息；`Category`＝类别。
`UNPARSEABLE`＝无法解析；输出中的 `unparseable replies`＝无法解析的回复数。

### 查看两种方法的分歧
两种方法预测不一致的地方，是分析报告的重要素材。动笔前先阅读这些案例。

In [ ]:
print(f'{"TRUE":<12}{"CLASSIFIER":<12}{"API":<12}TEXT')
for txt, truth, a, b in zip(X_test, y_test, pred_clf, pred_api):
    if a != b:
        print(f'{truth:<12}{a:<12}{b:<12}{txt[:60]}')

TRUE        CLASSIFIER  API         TEXT
behaviour   health_concernbehaviour   My cat asks to be petted but then bites and immediately asks
routine_careroutine_carebehaviour   How long should a newly arrived cat remain in one room befor
health_concernbehaviour   health_concernMy cat suddenly holds one paw off the ground after a nail tr
routine_carehealth_concernroutine_careMy cat has mild dental buildup but resists thorough brushing


表头对照：`TRUE`＝正确标签；`CLASSIFIER`＝传统分类器预测；`API`＝API 模型预测；`TEXT`＝输入文本（此处仅显示前 60 个字符）。

---
## 4 · 四项测量指标

**成本**是许多作业容易写得含糊的部分。分类器每次预测不产生调用费用，但你需要投入数据标注时间。API 每次预测只花费不足 1 美分，设置投入也很少。哪种更便宜，取决于**总共要分类多少条输入**，这正是作业要求讨论的规模问题。

In [ ]:
### REPLACE ME——在 openrouter.ai/models 核对所选模型当天价格，价格可能变化。
PRICE_IN_PER_M  = 0.15    # 每 1,000,000 个输入 Token 的美元价格。
PRICE_OUT_PER_M = 0.60    # 每 1,000,000 个输出 Token 的美元价格。

api_cost = TOKENS['in']/1e6*PRICE_IN_PER_M + TOKENS['out']/1e6*PRICE_OUT_PER_M
per_item = api_cost / max(1, len(X_test))

### REPLACE ME——如实估计投入小时数，这也是论证的一部分。
EFFORT = {'classifier': 'collecting, cleaning and labelling 58 training examples ≈ 4 h, then ~0 to run',
          'api':        'writing the prompt ≈ 0.1 h, no labelling,no traing data'}

print(f'{"":<14}{"ACCURACY":>10}{"COST / ITEM":>16}{"LATENCY":>12}   EFFORT')
print(f'{"baseline":<14}{majority[1]/len(DATA):>9.1%}{"$0":>16}{"~0 ms":>12}   none')
print(f'{"classifier":<14}{acc_clf:>9.1%}{"$0.00 (local)":>16}{clf_latency_ms:>9.2f} ms   {EFFORT["classifier"]}')
print(f'{"API":<14}{acc_api:>9.1%}{"$"+format(per_item,".6f"):>16}{api_latency_ms:>9.0f} ms   {EFFORT["api"]}')
print(f'\ntokens: {TOKENS["in"]} in / {TOKENS["out"]} out · this run cost about ${api_cost:.4f}')

for n in (1_000, 100_000, 10_000_000):
    print(f'  at {n:>10,} items/year -> API ≈ ${per_item*n:,.2f}   classifier ≈ $0 in API spend')

                ACCURACY     COST / ITEM     LATENCY   EFFORT
baseline          33.3%              $0       ~0 ms   none
classifier        87.5%   $0.00 (local)     0.12 ms   collecting, cleaning and labelling 58 training examples ≈ 4 h, then ~0 to run
API               93.8%       $0.000014      899 ms   writing the prompt ≈ 0.1 h, no labelling,no traing data

tokens: 2669 in / 98 out · this run cost about $0.0005
  at      1,000 items/year -> API ≈ $0.01   classifier ≈ $0 in API spend
  at    100,000 items/year -> API ≈ $1.43   classifier ≈ $0 in API spend
  at 10,000,000 items/year -> API ≈ $143.48   classifier ≈ $0 in API spend


### 投入与比较表文字对照
- 分类器投入：收集、清洗并标注 58 条训练样本，约 4 小时；之后运行投入约为零。
- API 投入：编写提示词约 0.1 小时，不进行标注、不需要训练数据。（原字段 `traing` 是 `training` 的拼写错误。）
- `ACCURACY`：准确率；`COST / ITEM`：每条成本；`LATENCY`：延迟；`EFFORT`：投入。
- `baseline`：基线；`classifier`：分类器；`none`：无；`local`：本地。
- `this run cost about`：本次运行费用约为；`items/year`：条/年；`classifier ≈ $0 in API spend`：分类器的 API 调用费用约为零。

*译注：上述投入是原文件中填写的估计，本次没有核实或修改。`TOKENS` 会累计连接测试等调用，重复运行也会累计；这里的单条成本口径需要在最终分析中说明。*

**仔细阅读上面的最后一段输出。** 原文说明：每年一千条时，API 成本通常很低，标注投入可能是更大的成本；达到一千万条时，成本比较可能改变，需要考虑为什么用付费模型完成 TF-IDF 能处理的任务。*你的成本交叉点在哪里？哪些条件发生变化后，你会改变选择？* 这就是约 400 词分析需要回答的问题。

---
## 5 · 轮到你了（YOUR TURN）

1. 将 `LABELS`、`DATA` 和 `SYSTEM` 替换为自己的领域内容，从头重新运行。
2. 检查两种方法是否都超过多数类基线。原文建议：如果两者都没有超过，类别可能存在重叠，应先检查类别定义。
3. 阅读预测分歧，选择**两个案例**，分析各方法为何出错。
4. 根据当天 OpenRouter 定价填写 `PRICE_*`；根据真实投入填写 `EFFORT`。
5. 提交比较表和约 400 词分析。

**值得尝试的调优选项（Knobs）：** 如果记录了变化，每项都可以成为报告中的分析内容。

- 将向量化器设为 `ngram_range=(1,2)`，同时统计单词与连续词对。
- 将 `test_size` 改为 `0.65`，减少训练集规模，观察分类器准确率下降多少。API 不训练，因此不会直接因训练样本减少而受影响。原文将这种差异视为支持 API 的重要依据。*译注：更换测试集也可能使 API 的测得准确率变化。*
- 在 `SYSTEM` 中加入两个完整示例，采用少样本提示（Few-shot，第 3 次课第 2 层），重新测量。注意，每次调用都会增加输入 Token。
- 将 `GEN_MODEL` 换成更便宜的模型。如果准确率保持不变，也是有价值的发现。

### 约 400 词分析必须包含什么

- 两种方法的比较表：准确率、成本、延迟、投入。
- 会部署哪种方法，以及**原因**，明确接受了哪项权衡。
- **达到什么规模时选择会改变**，附计算过程。
- 一项这里无法测量的内容，以及如何才能测量它。


In [ ]:
# 用于自己开展实验的草稿区。